# Pista C — evaluación off-policy y especificación de logging (C1–C2)

**19 de septiembre de 2026.** Corre la pista C de `docs/experimentos_productos.md` §4: el ensayo
en seco de la evaluación off-policy (C1), el dimensionado del experimento online y el
contrafactual retrospectivo sobre las empresas que rompieron caja en 2026 (C2).

La pista C existe porque **el dataset no trae registro de ofertas** (§2.1): no hay acciones
ofrecidas ni propensiones, así que hoy no se puede evaluar ninguna política con datos reales. Lo
que sí se puede hacer —y es lo que hace este cuaderno— es **simular el log que Embat tendría** si
instrumentara la recomendación mañana, correr encima los cuatro estimadores de `xray.ope` contra
una verdad que en el simulador sí se conoce, y medir cuánto se equivocan. De ahí sale el número
que importa para la especificación: cuántas ofertas hay que registrar y con cuánta exploración
para que el estimador diga algo.

Este cuaderno **importa `xray`, no define el pipeline**: el simulador y las historias son de
`xray.projection`, los candidatos, el objetivo y el MPC de `xray.policies`, y los estimadores de
`xray.ope`. Aquí solo se orquesta, se dibuja y se guarda.

Salidas en `artifacts/experiments/`: `C1_ope.csv`, `C1_ope.png`, `C1_power.csv`,
`C2_retrospectivo.csv` y `C_summary.json`. Lo que se lleva al equipo está en
`docs/logging_ofertas.md`.

**Advertencia que vale para todo el cuaderno:** el log es sintético y la política de
comportamiento la fabricamos nosotros, así que los números de C1 **no miden el valor de ningún
producto**: miden el error de los estimadores cuando la verdad se conoce. Es un ensayo del
instrumento, no una medición del mundo.

In [ ]:
import json
import time
import warnings
from dataclasses import replace

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import GroupKFold

from xray import ope, policies, projection
from xray.data import artifacts_dir, load
from xray.projection import FlowPool, SimConfig

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

T0 = time.time()
OUT = artifacts_dir() / "experiments"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 0
START = "2025-08"          # mes en el que arranca el log; 12 meses de ofertas por empresa
MONTHS = 12
EPSILONS = [0.1, 0.02, 0.2]  # 0,1 es la corrida principal; las otras dos son el barrido de ESS
SOFT_EPS = 0.05            # suavizado de la política objetivo
N_PATHS_MPC = 200          # caminos del MPC dentro del log (200 basta para ordenar acciones)
N_BOOT = 300
LAM_MULT, MU_MULT = 0.5, 0.25  # lam = 0,5 x mediana de cargos; mu = 0,25 x lam

# El espacio de acciones de la OPE es el **tipo** de producto, no el importe: 7 acciones fijas.
KINDS = ["none", "line_cover", "line_draw", "line_open", "factoring", "loan", "refinance"]
KIND_IDX = {k: i for i, k in enumerate(KINDS)}

# Vector de estado normalizado por la mediana de cargos (el mismo de la pista B, paso 6).
X_COLS = ["eom", "dip_min", "inflow_mean", "inflow_std", "line_room", "receivables",
          "loan_outstanding", "loan_rate", "debt_service", "months_history"]
print(f"salidas en {OUT}")

## Carga

`features.parquet` es la tabla del seam `features(company_id, month)`; `company_extras` añade lo
que esa tabla no lleva y el simulador necesita (límite y dispuesto de la línea, cartera elegible,
préstamo vivo). El cuaderno B cachea esa tabla en `B_extras.parquet`; si está, se reutiliza, y si
no se calcula y se guarda aparte (`C_extras.parquet`) para no pisarse con él.

In [ ]:
features = pd.read_parquet(artifacts_dir() / "features.parquet")
features["month"] = features["month"].astype(str)
months = sorted(features["month"].unique())
tables = load()

extras_b, extras_c = OUT / "B_extras.parquet", OUT / "C_extras.parquet"
if extras_b.exists():
    extras, extras_src = pd.read_parquet(extras_b), extras_b.name
elif extras_c.exists():
    extras, extras_src = pd.read_parquet(extras_c), extras_c.name
else:
    extras = projection.company_extras(
        tables["transactions"], tables["debt_products"], tables["banking_products"],
        tables["debt_schedule_config"], tables["invoices"], months, features=features,
    )
    extras.to_parquet(extras_c, index=False)
    extras_src = extras_c.name

cfg = SimConfig()
pool = FlowPool.fit(features)
hists = projection.histories(features, extras, cfg)
print(f"features {features.shape} · {months[0]} → {months[-1]} · "
      f"{features['company_id'].nunique()} empresas")
print(f"extras desde {extras_src} {extras.shape} · pool {pool.triplets.shape} · "
      f"historias {len(hists)}")

## Empresas del ensayo

Las mismas que el cuaderno B deja fuera del entrenamiento: el 20 % de los `group_id` del primer
pliegue de `GroupKFold(5)`. De esas empresas nos quedamos con las que tienen historia en
`2025-08` (≥ 6 meses) y datos hasta el final de la tabla.

El filtro estricto (último mes = `2026-08`) deja 126 empresas y el relajado (≥ 6 meses después de
`2025-08`) 130, así que **no se llega a las 150 del encargo ni relajando**: de las 253 empresas
del pliegue, solo 155 tienen historia reconstruida en `2025-08` y 136 llegan a 6 meses de
ventana. No se compensa cogiendo empresas de entrenamiento: el log tiene que salir de las mismas
empresas que la pista B, o los números no se pueden poner uno al lado del otro. Se sigue con las
que hay y el tamaño del log queda anotado en el resumen.

In [ ]:
comp = tables["companies"][["company_id", "group_id"]].drop_duplicates().sort_values("company_id")
comp = comp[comp["company_id"].isin(features["company_id"].unique())].reset_index(drop=True)
_, test_idx = next(GroupKFold(n_splits=5).split(comp, groups=comp["group_id"]))
held = comp.iloc[test_idx]["company_id"].tolist()
last_month = features.groupby("company_id")["month"].max()

with_history = [c for c in held if (c, START) in hists]
long_enough = [c for c in with_history if len(hists[(c, START)].outflows) >= 6]
strict = [c for c in long_enough if last_month.get(c) == months[-1]]
relaxed = [c for c in long_enough if last_month.get(c) >= "2026-02"]
# La mediana de cargos es la unidad de importe de toda la rejilla de acciones y el divisor de la
# normalización: con mediana 0 no hay ni acciones con importe ni escala, así que la empresa se cae.
selected = [c for c in relaxed if float(np.median(hists[(c, START)].outflows)) > 0]
print(f"pliegue retenido {len(held)} · con historia en {START} {len(with_history)} · "
      f"≥ 6 meses {len(long_enough)}")
print(f"estricto (último mes {months[-1]}) {len(strict)} · relajado (≥ 2026-02) {len(relaxed)} · "
      f"con mediana de cargos > 0 {len(selected)}")

## C1 — el log que Embat tendría

**Política de comportamiento.** Es el asesor de hoy con un poco de exploración encima: con
probabilidad `1 − ε` ofrece lo que dice `advisor_rules` (la regla de toda la vida: cubrir, abrir
línea, anticipar o refinanciar) y con probabilidad `ε` sortea uniformemente entre los tipos
elegibles en ese estado. La propensión con la que se registra la oferta es exactamente

    π_b(a | x) = (1 − ε)·1[a = regla] + ε / |elegibles|

y **se calcula en Python en el momento de la oferta**, no después: reconstruirla luego es
adivinar, y un IPS con propensiones adivinadas no es insesgado (esto es lo primero que se lleva a
`docs/logging_ofertas.md`).

**Acciones.** Siete tipos fijos. El importe concreto de cada tipo es el primer candidato de ese
tipo en `candidate_actions` (línea = un mes de cargos, factoring = 50 % de la cartera, préstamo =
un mes de cargos). Colapsar el importe dentro del tipo es una simplificación con precio: el MPC
elige otro importe dentro del mismo tipo en el 28 % de los estados, y esa diferencia queda fuera
de lo que la OPE ve. Se mide y se anota.

**Recompensa.** El objetivo realizado a 6 meses con la acción puesta, cambiado de signo:
`−[coste + λ·1(rotura) + μ·1(DSCR < 1,2)]` sobre **un** camino sorteado con la semilla
`[SEED, i, j]`, con `λ = 0,5 × mediana de cargos` y `μ = 0,25 λ`. Es el mismo modelo que optimiza
el MPC, evaluado en un sorteo concreto: un log real tampoco trae la esperanza, trae lo que pasó.

**El mundo avanza con la política de comportamiento.** El estado del mes `j+1` sale de simular un
mes con la acción *registrada* (semilla `[SEED, i, j, 7]`, independiente de la de la recompensa) y
`advance`. Es decir, el log es de **bucle cerrado**: la distribución de estados es la que genera
la política que registra, que es justo el sesgo que la OPE tiene que corregir.

In [ ]:
def fresh(hist):
    """Copia con arrays propios (espeja `policies._copy_history`, que es privada)."""
    return replace(hist, inflows=np.array(hist.inflows, dtype=float),
                   outflows=np.array(hist.outflows, dtype=float),
                   dips=np.array(hist.dips, dtype=float))


def state_vector(hist, med_out):
    """El estado que ve el modelo de recompensa, en unidades de mediana de cargos."""
    dips = np.asarray(hist.dips, dtype=float)
    inflows = np.asarray(hist.inflows, dtype=float)
    return [
        hist.eom / med_out,
        (float(np.min(dips)) if len(dips) else 0.0) / med_out,
        float(np.mean(inflows)) / med_out,
        float(np.std(inflows)) / med_out,
        max(0.0, hist.line_limit - hist.line_drawn) / med_out,
        hist.receivables / med_out,
        hist.loan_outstanding / med_out,
        0.0 if hist.loan_rate is None else float(hist.loan_rate),
        hist.debt_service_m / med_out,
        float(len(inflows)),
    ]


one = replace(cfg, n_paths=1)                 # un camino: el realizado
mpc_cfg = replace(cfg, n_paths=N_PATHS_MPC)   # la esperanza que optimiza el MPC


def build_log(epsilon, companies):
    """Log de ofertas de `MONTHS` meses por empresa, con la matriz de recompensas de toda acción.

    Devuelve `(log, rewards, eligible)`: `rewards[n, 7]` es la recompensa **normalizada** de cada
    tipo en ese estado (NaN si no es elegible) y `eligible[n, 7]` la máscara. Tener la matriz
    entera es lo que permite conocer la verdad de cualquier política objetivo sin volver a simular:
    en el simulador se puede evaluar cualquier acción sobre cualquier estado registrado, cosa que
    en un log real es imposible (y por eso hace falta la OPE).
    """
    rows, rewards, eligible_mask = [], [], []
    for i, company in enumerate(companies):
        hist = fresh(hists[(company, START)])
        scale = float(np.median(hist.outflows))  # el divisor es constante por empresa
        for j in range(MONTHS):
            med_out = float(np.median(hist.outflows))
            if med_out <= 0:  # sin unidad de importe no hay rejilla de acciones: se corta ahí
                break
            lam = LAM_MULT * med_out
            mu = MU_MULT * lam
            candidates = policies.candidate_actions(hist, cfg)
            first = {}
            for action in candidates:
                first.setdefault(action.kind, action)  # el primer candidato de cada tipo
            eligible = [k for k in KINDS if k in first]

            rules_kind = policies.advisor_rules(hist, cfg).kind
            if rules_kind not in first:  # no pasa hoy; si pasara, la propensión sería mentira
                rules_kind = "none"
            draw = np.random.default_rng([SEED, i, j, 3])
            explore = draw.random() >= 1.0 - epsilon
            kind = eligible[int(draw.integers(len(eligible)))] if explore else rules_kind
            propensity = (1.0 - epsilon) * (kind == rules_kind) + epsilon / len(eligible)

            reward = np.full(len(KINDS), np.nan)
            detail = {}
            for k in eligible:
                paths = projection.simulate(hist, first[k], one,
                                            rng=np.random.default_rng([SEED, i, j]),
                                            pool=pool, horizon=6)
                reward[KIND_IDX[k]] = -policies.objective(paths, lam, mu, cfg)
                detail[k] = (paths.expected_cost(), paths.breach_prob(),
                             paths.dscr_fail_prob(cfg.dscr_floor))
            recommendation = policies.mpc_recommend(
                hist, mpc_cfg, lam, mu, rng=np.random.default_rng([SEED, i, j, 1]), pool=pool)
            cost, breach, dscr = detail[kind]
            rows.append({
                "company_id": company, "i": i, "j": j, "month": hist.month,
                "action": KIND_IDX[kind], "kind": kind, "propensity": propensity,
                "explore": bool(explore),
                "reward": reward[KIND_IDX[kind]] / scale, "reward_eur": reward[KIND_IDX[kind]],
                "cost": cost / scale, "cost_eur": cost, "breach": breach, "dscr_fail": dscr,
                "rules_kind": rules_kind, "mpc_kind": recommendation.action.kind,
                "mpc_amount": float(recommendation.action.amount),
                "mpc_first_amount": float(first[recommendation.action.kind].amount),
                "n_eligible": len(eligible), "scale": scale, "lam": lam,
                **dict(zip(X_COLS, state_vector(hist, med_out))),
            })
            rewards.append(reward / scale)
            eligible_mask.append([k in first for k in KINDS])

            # El mundo avanza con lo que se ofreció (bucle cerrado), con otra corriente de sorteos.
            step = projection.simulate(hist, first[kind], one,
                                       rng=np.random.default_rng([SEED, i, j, 7]),
                                       pool=pool, horizon=1)
            hist = projection.advance(hist, step, first[kind], k=0, cfg=one)
    return pd.DataFrame(rows), np.array(rewards), np.array(eligible_mask)


logs = {}
for epsilon in EPSILONS:
    t = time.time()
    logs[epsilon] = build_log(epsilon, selected)
    log = logs[epsilon][0]
    print(f"ε = {epsilon}: {len(log)} ofertas · {log['company_id'].nunique()} empresas · "
          f"exploración {log['explore'].mean():.3f} · {time.time() - t:.1f} s")
print("\nmezcla registrada (ε = 0,1):", logs[0.1][0]["kind"].value_counts().to_dict())
print("mezcla de la regla:      ", logs[0.1][0]["rules_kind"].value_counts().to_dict())
print("mezcla del MPC:          ", logs[0.1][0]["mpc_kind"].value_counts().to_dict())

## La política objetivo y la verdad

**Objetivo determinista:** el tipo que recomienda `mpc_recommend` con 200 caminos, ejecutado al
importe del primer candidato de ese tipo. `target_probs` es una one-hot, así que el peso de
importancia vale `1/π_b` en las filas donde el asesor ofreció justo eso y **0 en todas las
demás**: la OPE de una política determinista solo se apoya en las coincidencias.

**Objetivo suavizado:** la misma recomendación con `ε = 0,05` repartido entre los tipos
elegibles. Es la versión que de verdad se desplegaría (una política que nunca explora no se puede
volver a evaluar el mes que viene) y, de paso, da solape con acciones que la one-hot descarta.

**La verdad.** Como el log sale del simulador, la recompensa de *cualquier* tipo sobre *cualquier*
estado registrado está en la matriz `rewards`, con la misma semilla `[SEED, i, j]`. El valor real
de la política objetivo es entonces `media_x Σ_a π_e(a|x)·r(x, a)`, sin estimar nada. Esa es la
vara: cada estimador se compara contra ella.

In [ ]:
def target_matrices(log, mask):
    """`(one-hot del MPC, versión suavizada)` como matrices (n, 7)."""
    n = len(log)
    rows = np.arange(n)
    mpc = np.array([KIND_IDX[k] for k in log["mpc_kind"]])
    # El MPC enumera `candidate_actions`, así que su tipo siempre está entre los elegibles; si no
    # lo estuviera, la verdad sumaría un NaN como 0 y saldría un número que no es de nadie.
    assert mask[rows, mpc].all(), "el MPC recomendó un tipo fuera de la máscara de elegibles"
    hard = np.zeros((n, len(KINDS)))
    hard[rows, mpc] = 1.0
    soft = np.where(mask, SOFT_EPS / mask.sum(axis=1, keepdims=True), 0.0)
    soft[rows, mpc] += 1.0 - SOFT_EPS
    return hard, soft


def truth(probs, rewards):
    """Valor exacto de la política: media de Σ_a π_e(a|x)·r(x, a) (los NaN pesan 0)."""
    return float(np.nansum(probs * np.nan_to_num(rewards, nan=0.0), axis=1).mean())


summary_c1 = {}
for epsilon in EPSILONS:
    log, rewards, mask = logs[epsilon]
    hard, soft = target_matrices(log, mask)
    match = float((log["kind"] == log["mpc_kind"]).mean())
    summary_c1[epsilon] = {
        "n_offers": int(len(log)), "n_companies": int(log["company_id"].nunique()),
        "explore_share": round(float(log["explore"].mean()), 4),
        "match_rate": round(match, 4),
        "rules_vs_mpc": round(float((log["rules_kind"] == log["mpc_kind"]).mean()), 4),
        "mpc_amount_differs": round(float((log["mpc_amount"] != log["mpc_first_amount"]).mean()), 4),
        "logged_mix": log["kind"].value_counts().to_dict(),
        "mpc_mix": log["mpc_kind"].value_counts().to_dict(),
        "behaviour_value": round(float(log["reward"].mean()), 4),
        "behaviour_value_eur": round(float(log["reward_eur"].mean()), 2),
        "truth_hard": round(truth(hard, rewards), 4),
        "truth_soft": round(truth(soft, rewards), 4),
    }
    print(f"ε = {epsilon}: coincidencia log = MPC {match:.3f} · verdad determinista "
          f"{truth(hard, rewards):.4f} · suavizada {truth(soft, rewards):.4f} · "
          f"comportamiento {log['reward'].mean():.4f}")
print(f"\nel MPC elige un importe distinto del primer candidato en el "
      f"{summary_c1[0.1]['mpc_amount_differs']:.0%} de los estados (ε = 0,1)")

## Los cuatro estimadores

`ips` y `snips` reponderan por importancia, `dm` evalúa un modelo de recompensa LightGBM
cross-fitted y `dr` combina los dos. Intervalos por bootstrap de 300 réplicas (percentil 95 %) y
`effective_sample_size` al lado: si el ESS cae muy por debajo de n, el log no solapa con la
política objetivo y el número no vale, por mucho que el intervalo salga estrecho.

Se estima en **las dos escalas**: la recompensa normalizada por la mediana de cargos de cada
empresa (comparable entre una pyme de 50 K y una de 5 M) y la recompensa en euros brutos. Los
estimadores son lineales en la recompensa, así que cada escala tiene su propia verdad y el error
relativo es lo único que se puede comparar entre las dos.

In [ ]:
ESTIMATORS = {"ips": ope.ips, "snips": ope.snips, "dm": ope.dm, "dr": ope.dr}


def estimate_all(log, probs, q_hat):
    """Punto, intervalo y ESS de los cuatro estimadores sobre un log y una política objetivo."""
    out = {}
    ess = ope.effective_sample_size(log, probs)
    for name, fn in ESTIMATORS.items():
        kw = {"q_hat": q_hat} if name in ("dm", "dr") else {}
        try:
            point = fn(log, probs, **kw)
            lo, hi = ope.bootstrap_ci(fn, log, probs, n_boot=N_BOOT, seed=SEED, **kw)
        except ValueError as exc:  # p. ej. Σw = 0 en una réplica: se anota, no se disimula
            point, lo, hi = np.nan, np.nan, np.nan
            print(f"  {name}: {exc}")
        out[name] = {"estimate": point, "ci_lo": lo, "ci_hi": hi, "ess": ess}
    return out


rows = []
for epsilon in EPSILONS:
    log, rewards, mask = logs[epsilon]
    hard, soft = target_matrices(log, mask)
    euros = log.assign(reward=log["reward_eur"])
    scales = {
        "normalizado": (log, ope.fit_reward_model(log, X_COLS, len(KINDS), seed=SEED), rewards),
        "euros": (euros, ope.fit_reward_model(euros, X_COLS, len(KINDS), seed=SEED),
                  rewards * log["scale"].to_numpy()[:, None]),
    }
    for scale_name, (frame, q_hat, scale_rewards) in scales.items():
        for target_name, probs in (("determinista", hard), ("suavizado", soft)):
            value = truth(probs, scale_rewards)
            for name, res in estimate_all(frame, probs, q_hat).items():
                rows.append({
                    "epsilon": epsilon, "scale": scale_name, "target": target_name,
                    "estimator": name, "n": len(frame), "ess": res["ess"],
                    "estimate": res["estimate"], "ci_lo": res["ci_lo"], "ci_hi": res["ci_hi"],
                    "truth": value, "error": res["estimate"] - value,
                    "rel_error": (res["estimate"] - value) / abs(value) if value else np.nan,
                    "covers": bool(res["ci_lo"] <= value <= res["ci_hi"]),
                    "behaviour_value": float(frame["reward"].mean()),
                })
ope_table = pd.DataFrame(rows)
ope_table.to_csv(OUT / "C1_ope.csv", index=False)
print(f"{OUT / 'C1_ope.csv'} · {len(ope_table)} filas\n")
view = ope_table.query("scale == 'normalizado'")[
    ["epsilon", "target", "estimator", "ess", "estimate", "ci_lo", "ci_hi", "truth", "rel_error",
     "covers"]]
print(view.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

### Qué sale

Cuatro cosas, y las cuatro van a la especificación:

1. **El ESS es el número que manda.** Sobre 1.472 ofertas registradas, el tamaño efectivo del log
   frente al objetivo determinista es 125 con ε = 0,1, **20** con ε = 0,02 y 204 con ε = 0,2: del
   1 % al 14 % del log. Solo cuentan las filas en las que el asesor ofreció justo lo que el MPC
   habría ofrecido, y bajar ε no las aumenta —la coincidencia se queda en el 51–55 %—, solo
   multiplica el peso de las pocas exploradas. Menos exploración es **peor** evaluación.
2. **IPS lo decide una fila.** La tabla de abajo ordena las filas por su contribución a IPS: con
   ε = 0,1 la primera se lleva el 81 % del estimador y las cinco primeras el 92 %. Esa primera es
   una oferta de préstamo sorteada por exploración (propensión 1/30) a una empresa ya en
   descubierto, con una recompensa realizada de 38 veces su mediana de cargos en negativo. Un log
   real tendrá colas así. SNIPS no la arregla (autonormaliza el peso, no la recompensa: se va aún
   más lejos, a −1,18) y DR apenas la recorta (−0,90), porque el residuo `r − q̂` de esa fila
   sigue entrando multiplicado por 30.
3. **DM acierta casi siempre y no se le puede creer.** El modelo de recompensa está entrenado
   sobre el mismo simulador que genera la verdad, así que aquí tiene la especificación
   exactamente bien y su error relativo es del 2 %. Con datos reales no la tendrá, y su intervalo
   de bootstrap **no incluye el sesgo de especificación**: es el intervalo más estrecho de la
   tabla y el único que puede estar centrado en el sitio equivocado.
4. **En euros brutos no hay evaluación que valga.** La misma tabla en euros (`scale == "euros"` en
   el CSV) da un valor de comportamiento de −11,8 M€ y una verdad de −3,6 M€, y los intervalos de
   IPS y SNIPS **no cubren la verdad en ninguna ε**: las empresas de cargos gigantes dominan la
   media y casi ninguna de sus filas coincide con el MPC, así que el estimador reponderado se las
   salta. Por eso se normaliza por la mediana de cargos antes de estimar (y por eso DR, que arrastra
   el término de modelo, es el único que aguanta en euros).

In [ ]:
def ips_contributions(epsilon):
    """Contribución de cada fila al IPS del objetivo determinista: `w·r/n`."""
    log, _, mask = logs[epsilon]
    hard, _ = target_matrices(log, mask)
    w = np.where(hard[np.arange(len(log)), log["action"].to_numpy(int)] > 0,
                 1.0 / log["propensity"].to_numpy(), 0.0)
    return pd.DataFrame({
        "company_id": log["company_id"], "month": log["month"], "kind": log["kind"],
        "explore": log["explore"], "propensity": log["propensity"].round(4), "w": w.round(2),
        "reward": log["reward"].round(3), "contrib_ips": (w * log["reward"] / len(log)).round(4),
    }), w


concentration = {}
for epsilon in EPSILONS:
    contrib, w = ips_contributions(epsilon)
    ordered = contrib.reindex(contrib["contrib_ips"].abs().sort_values(ascending=False).index)
    concentration[epsilon] = {
        "max_weight": float(w.max()), "n_positive": int((w > 0).sum()),
        "median_positive_weight": float(np.median(w[w > 0])),
        "share_top1": float(ordered["contrib_ips"].iloc[0] / contrib["contrib_ips"].sum()),
        "share_top5": float(ordered["contrib_ips"].head(5).sum() / contrib["contrib_ips"].sum()),
    }
    print(f"ε = {epsilon}: peso máximo {w.max():.1f} · filas con peso > 0 {int((w > 0).sum())} · "
          f"mediana de esos pesos {np.median(w[w > 0]):.2f} · la primera fila explica el "
          f"{concentration[epsilon]['share_top1']:.0%} del IPS y las cinco primeras el "
          f"{concentration[epsilon]['share_top5']:.0%}")

contrib, weights = ips_contributions(0.1)
top = contrib.reindex(contrib["contrib_ips"].abs().sort_values(ascending=False).index).head(6)
share_top1 = concentration[0.1]["share_top1"]
print(f"\nlas seis filas que más pesan en el IPS con ε = 0,1:\n{top.to_string(index=False)}")

In [ ]:
fig, axes = plt.subplots(2, len(EPSILONS), figsize=(4.6 * len(EPSILONS), 7.6), sharey="row")
palette = {"ips": "#c0392b", "snips": "#e67e22", "dm": "#2980b9", "dr": "#27ae60"}
for col, epsilon in enumerate(EPSILONS):
    for row, target_name in enumerate(["determinista", "suavizado"]):
        ax = axes[row, col]
        sub = ope_table.query("epsilon == @epsilon and scale == 'normalizado' and "
                              "target == @target_name").reset_index(drop=True)
        for xi, r in sub.iterrows():
            err = np.abs([[r["estimate"] - r["ci_lo"]], [r["ci_hi"] - r["estimate"]]])
            ax.errorbar(xi, r["estimate"], yerr=err, fmt="o", ms=8, lw=2.6, capsize=6,
                        capthick=2.0, color=palette[r["estimator"]], zorder=3)
        value = sub["truth"].iloc[0]
        ax.axhline(value, color="#111111", ls="--", lw=1.4, zorder=1, label="verdad")
        ax.axhline(sub["behaviour_value"].iloc[0], color="#888888", ls=":", lw=1.4, zorder=1,
                   label="comportamiento")
        ax.set_xticks(np.arange(len(sub)))
        ax.set_xticklabels(sub["estimator"])
        ax.set_xlim(-0.6, len(sub) - 0.4)
        ax.set_title(f"ε = {epsilon} · {target_name}\nESS {sub['ess'].iloc[0]:,.0f} / n "
                     f"{sub['n'].iloc[0]:,} · verdad {value:.3f}", fontsize=10)
        ax.grid(axis="y", alpha=0.25)
        if col == 0:
            ax.set_ylabel("valor de la política\n(recompensa / mediana de cargos)")
        if row == 0 and col == 0:
            ax.legend(loc="lower left", fontsize=8)
fig.suptitle("C1 · estimadores off-policy frente a la verdad del simulador "
             f"(objetivo MPC, {len(selected)} empresas × {MONTHS} meses)", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(OUT / "C1_ope.png", dpi=140)
plt.close(fig)
print(f"{OUT / 'C1_ope.png'}")

## Potencia: cuántas ofertas hay que registrar

Dos preguntas distintas y dos respuestas distintas.

**Binaria** (aceptación de la oferta, o rotura de caja a 6 meses): `power_two_proportions(0.05,
0.03)` da el tamaño por brazo para distinguir un 5 % de un 3 % con α = 0,05 y potencia 0,8;
`n_offers_for_power` lo traduce a ofertas registradas cuando solo una fracción ε se aleatoriza
(`2·n/ε`). Los meses de registro salen a una oferta por empresa y mes, con las 1.286 empresas del
dataset y con un piloto de 400.

**Continua** (coste financiero): `n = 2·(z_{1−α/2} + z_{potencia})²·σ²/Δ²` por brazo, con σ la
desviación típica de la recompensa normalizada del log y Δ = 10 % del coste normalizado medio. La
fila principal es la del encargo; debajo van dos variantes que enseñan de dónde viene el número:
usar la σ del **coste** en vez de la del objetivo (el objetivo lleva dentro el salto de λ de la
rotura) y winsorizar la recompensa al percentil 1–99. La distancia entre las tres es el precio de
la cola.

**La conclusión del bloque es un «no».** Con σ = 2,77 y Δ = 0,021 meses de cargos hacen falta
261.697 ofertas por brazo (5,2 millones registradas con ε = 0,1: 4.070 meses con las 1.286
empresas). Recortando la cola al percentil 99 siguen siendo 20.926 por brazo. Un efecto del 10 %
sobre el coste financiero **no se mide en un piloto**; el experimento online se dimensiona sobre
el resultado binario —aceptación de la oferta o rotura a 6 meses—, que cuesta 1.504 por brazo, y
el coste se sigue por OPE con el log completo, no con un contraste de medias.

In [ ]:
Z = norm.ppf(0.975) + norm.ppf(0.8)
POWER_EPS = [0.05, 0.1, 0.2]
FLEETS = {"dataset_1286": 1286, "piloto_400": 400}

log = logs[0.1][0]
sigma_reward = float(log["reward"].std())
sigma_cost = float(log["cost"].std())
clip = log["reward"].clip(*np.percentile(log["reward"], [1, 99]))
sigma_winsor = float(clip.std())
delta = 0.10 * float(log["cost"].mean())

n_binary = ope.power_two_proportions(0.05, 0.03)
continuous = {
    "coste_sigma_objetivo": sigma_reward,
    "coste_sigma_coste": sigma_cost,
    "coste_sigma_winsorizada": sigma_winsor,
}

rows = []
for label, n_arm in [("conversion_5_vs_3", n_binary)] + [
        (k, int(np.ceil(2 * Z**2 * s**2 / delta**2))) for k, s in continuous.items()]:
    for epsilon in POWER_EPS:
        offers = int(np.ceil(2 * n_arm / epsilon))
        row = {"outcome": label, "n_per_arm": int(n_arm), "epsilon": epsilon, "n_offers": offers,
               "sigma": continuous.get(label, np.nan),
               "delta": delta if label in continuous else 0.02}
        for name, fleet in FLEETS.items():
            row[f"months_{name}"] = round(offers / fleet, 1)
        rows.append(row)
power = pd.DataFrame(rows)
power.to_csv(OUT / "C1_power.csv", index=False)
print(f"{OUT / 'C1_power.csv'}\n")
print(f"σ objetivo {sigma_reward:.3f} · σ coste {sigma_cost:.3f} · σ winsorizada "
      f"{sigma_winsor:.3f} · coste medio {log['cost'].mean():.4f} → Δ {delta:.4f} "
      "(en unidades de mediana de cargos)")
print(power.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
print("\n`delta` es 0,02 en la fila binaria (5 % − 3 %, diferencia absoluta de tasas) y "
      f"{delta:.4f} en las continuas (10 % del coste medio, en meses de cargos).")
print(f"comprobación: n_offers_for_power(0.05, 0.03, ε) = "
      f"{[ope.n_offers_for_power(0.05, 0.03, e) for e in POWER_EPS]}")

## C2 — el contrafactual retrospectivo

La pregunta del asesor en la demo: *de las empresas que se quedaron sin caja en 2026, ¿a cuántas
les habríamos dicho algo tres o seis meses antes, y cuánto habría bajado la probabilidad de
rotura?*

La entrada en rotura es el **primer** mes ≥ `2026-01` con `min_balance_eur < 0` dos meses
seguidos y ≥ 0 en los dos anteriores (dos meses seguidos para no contar un pico de un día, y dos
meses limpios antes para que sea una entrada y no la continuación de un problema viejo). En
`t − 3` y `t − 6` se corre `mpc_recommend` con los 500 caminos de la ficha en pantalla y se
compara `breach_prob` de lo recomendado contra `breach_prob_none`.

Esto **no es una evaluación**: es el motor mirando hacia atrás con la información que había en
`t − k`. Que baje la probabilidad simulada de rotura no demuestra que la rotura real se hubiera
evitado; lo que demuestra es que la señal estaba en los datos con tres y con seis meses de
antelación. Y hay que leer el ΔP con el mecanismo delante: en el simulador la póliza **cubre el
descubierto por construcción** mientras quede disponible, así que en cuanto el límite da de sí el
ΔP es casi todo el `breach_prob_none`. Lo que C2 mide es «había señal y había producto con
capacidad», no «la rotura no habría ocurrido».

In [ ]:
balances = features.pivot_table(index="company_id", columns="month", values="min_balance_eur")
balances = balances.reindex(columns=months)
values = balances.to_numpy(float)
negative = np.isfinite(values) & (values < 0)
clean = np.isfinite(values) & (values >= 0)

entries = []
for row, company in enumerate(balances.index):
    for j in range(2, len(months) - 1):
        if months[j] < "2026-01":
            continue
        if negative[row, j] and negative[row, j + 1] and clean[row, j - 1] and clean[row, j - 2]:
            entries.append((company, months[j]))
            break
print(f"{len(entries)} entradas en rotura en 2026 "
      f"({pd.Series([m for _, m in entries]).value_counts().sort_index().to_dict()})")

rows = []
for k, (company, entry) in enumerate(entries):
    for lag in (3, 6):
        month = str(pd.Period(entry, freq="M") - lag)
        hist = hists.get((company, month))
        if hist is None:
            continue
        med_out = float(np.median(hist.outflows))
        if med_out <= 0:
            continue
        lam = LAM_MULT * med_out
        recommendation = policies.mpc_recommend(hist, cfg, lam, MU_MULT * lam,
                                                rng=np.random.default_rng([SEED, k, lag]),
                                                pool=pool)
        action = recommendation.action
        amount_eur = (action.amount * hist.receivables if action.kind == "factoring"
                      else action.amount)
        rows.append({
            "company_id": company, "entry_month": entry, "lag": lag, "month": month,
            "kind": action.kind, "amount": float(action.amount), "amount_eur": float(amount_eur),
            "rate": None if action.rate is None else float(action.rate),
            "expected_cost": recommendation.expected_cost,
            "cost_over_outflow": recommendation.expected_cost / med_out,
            "breach_prob": recommendation.breach_prob,
            "breach_prob_none": recommendation.breach_prob_none,
            "delta_breach": recommendation.breach_prob_none - recommendation.breach_prob,
            "dscr_fail_prob": recommendation.dscr_fail_prob,
            "objective": recommendation.objective,
            "med_outflow_eur": med_out, "months_history": int(len(hist.outflows)),
            "n_candidates": len(recommendation.alternatives),
        })
retro = pd.DataFrame(rows)
retro.to_csv(OUT / "C2_retrospectivo.csv", index=False)
print(f"{OUT / 'C2_retrospectivo.csv'} · {len(retro)} filas "
      f"({retro['company_id'].nunique()} empresas)\n")

acts = retro[retro["kind"] != "none"]
print(retro.groupby("lag").agg(
    n=("company_id", "size"),
    share_accion=("kind", lambda s: float((s != "none").mean())),
    delta_breach_medio=("delta_breach", "mean"),
    breach_none_medio=("breach_prob_none", "mean"),
    coste_mediano_eur=("expected_cost", "median"),
    coste_mediano_rel=("cost_over_outflow", "median"),
).round(4).to_string())
print("\nmezcla de recomendaciones:", retro["kind"].value_counts().to_dict())
print(f"ΔP(rotura) medio cuando recomienda algo: {acts['delta_breach'].mean():.4f} "
      f"(n = {len(acts)}); filas con ΔP > 0: {int((retro['delta_breach'] > 0).sum())}")
# La media del coste en euros no se publica: la mediana de cargos va de 0,01 € a 867 M€ en esta
# tabla, así que una sola empresa se lleva la media. En pantalla, el coste relativo.
print(f"coste esperado: mediana {retro['expected_cost'].median():,.0f} € · media "
      f"{retro['expected_cost'].mean():,.0f} € (la media es de una empresa, no de la muestra) · "
      f"mediana relativa {retro['cost_over_outflow'].median():.4f} meses de cargos")
print("\ntop 5 por ΔP(rotura), desempatado por coste — las de la demo:")
top5 = retro.sort_values(["delta_breach", "expected_cost"], ascending=[False, True]).head(5)[
    ["company_id", "entry_month", "lag", "kind", "amount_eur", "expected_cost",
     "breach_prob_none", "breach_prob", "delta_breach"]]
print(top5.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

## Lo que se lleva a `docs/logging_ofertas.md`

1. **Tres tablas**: `offers` (con el estado, los candidatos, la recomendación y **la propensión**),
   `responses` (lo que hizo el asesor y lo que dijo el cliente) y `outcomes` (lo que pasó seis
   meses después, rellenado por el job mensual de `xray-score`).
2. **La propensión se escribe en el momento de la oferta, en Python.** El LLM no calcula: Eve
   redacta sobre el JSON, y el JSON trae ya la propensión y los candidatos.
3. **ε = 0,1 como punto de partida.** Con ε = 0,02 el ESS se hunde de 125 a 20 (los pocos casos
   explorados llegan a pesar 200 veces) y con ε = 0,2 el asesor ofrece una de cada cinco veces algo
   que no haría.
4. **SNIPS primero, DR a partir de ~500 resultados observados**, y el ESS siempre al lado del
   número: si cae por debajo de ~10 % de n, el resultado no se presenta.
5. **Los números de potencia** de `C1_power.csv`: 1.504 ofertas por brazo para 5 % contra 3 %, o
   30.080 ofertas registradas con ε = 0,1 — 23 meses a una oferta por empresa y mes con las 1.286
   empresas del dataset, o 75 meses con un piloto de 400. El coste financiero no se dimensiona:
   con la σ del log haría falta un piloto de años.
6. **La recompensa se normaliza por la mediana de cargos** antes de estimar nada, y el resultado
   binario se registra tal cual. En euros brutos el estimador se lo comen tres empresas.

## Resumen

`C_summary.json` recoge el tamaño del log y la mezcla de acciones por ε, la verdad y las
estimaciones con intervalo y ESS de los cuatro estimadores, la tabla de potencia, los recuentos de
C2 y el tiempo de reloj del cuaderno entero (`wall_time_s`). **Es la última celda a propósito**:
el resumen tiene que medir la ejecución completa, así que nada puede correr después de él.

In [ ]:
def clean_number(value):
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return None
    return round(float(value), 6)


estimates = {}
for epsilon in EPSILONS:
    per_scale = {}
    for scale_name in ("normalizado", "euros"):
        per_target = {}
        for target_name in ("determinista", "suavizado"):
            sub = ope_table.query("epsilon == @epsilon and scale == @scale_name and "
                                  "target == @target_name")
            per_target[target_name] = {
                "truth": clean_number(sub["truth"].iloc[0]),
                "behaviour_value": clean_number(sub["behaviour_value"].iloc[0]),
                "ess": clean_number(sub["ess"].iloc[0]),
                "by_estimator": {
                    r["estimator"]: {
                        "estimate": clean_number(r["estimate"]), "ci_lo": clean_number(r["ci_lo"]),
                        "ci_hi": clean_number(r["ci_hi"]), "error": clean_number(r["error"]),
                        "rel_error": clean_number(r["rel_error"]), "covers": bool(r["covers"]),
                    } for _, r in sub.iterrows()
                },
            }
        per_scale[scale_name] = per_target
    estimates[str(epsilon)] = {**summary_c1[epsilon], "by_scale": per_scale}

summary = {
    "generated_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "setup": {
        "companies_held_out": int(len(held)), "companies_used": int(len(selected)),
        "start_month": START, "months": MONTHS, "kinds": KINDS,
        "n_paths_mpc": N_PATHS_MPC, "n_paths_reward": 1, "n_boot": N_BOOT,
        "lam_multiplier": LAM_MULT, "mu_multiplier": MU_MULT, "soft_epsilon": SOFT_EPS,
        "extras_source": extras_src, "seed": SEED,
    },
    "c1": estimates,
    "c1_concentration": {
        str(epsilon): {k: clean_number(v) if k != "n_positive" else int(v)
                       for k, v in values.items()}
        for epsilon, values in concentration.items()
    },
    "c1_ips_top_row": {
        "share_of_ips": clean_number(share_top1),
        "max_weight": clean_number(float(weights.max())),
        "row": {k: (v if isinstance(v, (str, bool)) else clean_number(v))
                for k, v in top.iloc[0].to_dict().items()},
    },
    "power": {
        "sigma_objetivo": clean_number(sigma_reward), "sigma_coste": clean_number(sigma_cost),
        "sigma_winsorizada": clean_number(sigma_winsor), "delta": clean_number(delta),
        "n_per_arm_binary": int(n_binary),
        "rows": [{k: (v if isinstance(v, str) else clean_number(v)) for k, v in r.items()}
                 for r in power.to_dict("records")],
    },
    "c2": {
        "n_entries": int(len(entries)), "n_rows": int(len(retro)),
        "n_companies": int(retro["company_id"].nunique()),
        "share_action": clean_number(float((retro["kind"] != "none").mean())),
        "mean_delta_breach": clean_number(float(retro["delta_breach"].mean())),
        "mean_delta_breach_when_action": clean_number(float(acts["delta_breach"].mean())),
        "mean_breach_prob_none": clean_number(float(retro["breach_prob_none"].mean())),
        "median_expected_cost_eur": clean_number(float(retro["expected_cost"].median())),
        "median_cost_over_outflow": clean_number(float(retro["cost_over_outflow"].median())),
        "by_lag": {
            str(lag): {
                "n": int(len(g)), "share_action": clean_number(float((g["kind"] != "none").mean())),
                "mean_delta_breach": clean_number(float(g["delta_breach"].mean())),
                "mean_breach_prob_none": clean_number(float(g["breach_prob_none"].mean())),
                "median_expected_cost": clean_number(float(g["expected_cost"].median())),
                "median_cost_over_outflow": clean_number(float(g["cost_over_outflow"].median())),
            } for lag, g in retro.groupby("lag")
        },
        "kind_mix": retro["kind"].value_counts().to_dict(),
        "top5": [{k: (v if isinstance(v, str) else clean_number(v)) for k, v in r.items()}
                 for r in top5.to_dict("records")],
    },
    "wall_time_s": round(time.time() - T0, 1),
}
path = OUT / "C_summary.json"
path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{path} · {path.stat().st_size / 1024:.1f} KB · "
      f"tiempo total {summary['wall_time_s']} s\n")
print(json.dumps(summary, indent=2, ensure_ascii=False))